> 在线讲义：[第 16 章 · 跨本体：不同的机器人，怎么排进同一张动作表](https://wikieden.github.io/vla-from-scratch/lessons/16-cross-embodiment.html) · 本书《VLA 从零到 π0》

# 第 16 章 · 跨本体：不同的机器人，怎么排进同一张动作表

配套讲义：仓库里 `vla-deep-dive/notes/lessons/16-cross-embodiment.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 2 步 | `unified_slots(robot)`、`fill_unified(values, robot)` | 单臂 7 / 128 格（0–5、10）；ALOHA 14 / 128 格 |
| 第 2 步 | `masked_mse(pred, target, mask)` | 空格不算损失 |
| 第 3 步 | `soft_prompt_params(30, 32, 1024)` | 983,040 |
| 第 3 步 | `rot6d_to_matrix(x)` | 正交矩阵，行列式 = 1 |

统一空间布局取自 rdt-1b-2410.07864.txt 附录 C 表 4；软提示尺寸取自 LeRobot xvla/configuration_xvla.py。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

In [ ]:
UNIFIED_DIM = 128
LAYOUT = {
    "right_arm_joint_pos": (0, 10),
    "right_gripper_pos": (10, 15),
    "right_arm_joint_vel": (15, 25),
    "right_gripper_vel": (25, 30),
    "right_eef_pos": (30, 33),
    "right_eef_6d": (33, 39),
    "left_arm_joint_pos": (50, 60),
    "left_gripper_pos": (60, 65),
    "base_lin_vel": (100, 102),
    "base_ang_vel": (102, 103),
}
ROBOTS = {
    "6 自由度单臂": {"right_arm_joint_pos": 6, "right_gripper_pos": 1},
    "ALOHA 双臂": {"right_arm_joint_pos": 6, "right_gripper_pos": 1, "left_arm_joint_pos": 6, "left_gripper_pos": 1},
}

**函数 `unified_slots`**：一种机器人的动作按物理意义填进 128 格时，用到哪些格子。

In [ ]:
def unified_slots(robot):
    """一种机器人的动作按物理意义填进 128 格时，用到哪些格子。"""
    slots = []
    for key, n in ROBOTS[robot].items():
        start, end = LAYOUT[key]
        assert n <= end - start
        slots += list(range(start, start + n))
    return slots

**函数 `fill_unified`**：把原始动作向量按含义填进 128 格，返回 (值, 掩码)。

In [ ]:
def fill_unified(values, robot):
    """把原始动作向量按含义填进 128 格，返回 (值, 掩码)。"""
    slots = unified_slots(robot)
    assert len(values) == len(slots)
    vec, mask = np.zeros(UNIFIED_DIM), np.zeros(UNIFIED_DIM)
    vec[slots], mask[slots] = values, 1.0
    return vec, mask

**函数 `masked_mse`**：只在有数的格子上算均方误差。

In [ ]:
def masked_mse(pred, target, mask):
    """只在有数的格子上算均方误差。"""
    pred, target, mask = (np.asarray(a, float) for a in (pred, target, mask))
    return float((mask * (pred - target) ** 2).sum() / mask.sum())

**函数 `soft_prompt_params`**：X-VLA 软提示的参数量：一张 num_domains × (len_prompts · hidden) 的查表。

In [ ]:
def soft_prompt_params(num_domains, len_prompts, hidden):
    """X-VLA 软提示的参数量：一张 num_domains × (len_prompts · hidden) 的查表。"""
    return num_domains * len_prompts * hidden

**函数 `rot6d_to_matrix`**：6D 旋转（两列）→ 旋转矩阵：Gram–Schmidt 正交化，再叉乘补第三列。

In [ ]:
def rot6d_to_matrix(x):
    """6D 旋转（两列）→ 旋转矩阵：Gram–Schmidt 正交化，再叉乘补第三列。"""
    a1, a2 = np.asarray(x[:3], float), np.asarray(x[3:6], float)
    b1 = a1 / np.linalg.norm(a1)
    u2 = a2 - (b1 @ a2) * b1
    b2 = u2 / np.linalg.norm(u2)
    return np.stack([b1, b2, np.cross(b1, b2)], axis=1)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 按物理意义填 128 格")
for robot in ROBOTS:
    s = unified_slots(robot)
    print(f"  {robot}：用 {len(s)} / {UNIFIED_DIM} 格（{len(s) / UNIFIED_DIM:.1%}），格号 {s}")
vec, mask = fill_unified([0.1, -0.2, 0.3, 0.0, 0.5, -0.1, 1.0], "6 自由度单臂")
pred = vec + 0.1 * mask  # 有数的格子各差 0.1
pred[127] = 9.0  # 空格子上乱猜也不扣分
print(f"  掩码损失 = {masked_mse(pred, vec, mask):.4f}（空格第 127 格猜 9.0 不算）")

In [ ]:
print("\n第三步 X-VLA 软提示与 6D 旋转")
print(f"  30 个域 × 32 个提示 × 1024 维 = {soft_prompt_params(30, 32, 1024):,} 个参数")
r = rot6d_to_matrix([1, 0.2, 0, 0.1, 1, 0.3])
print(f"  6D → 矩阵，RᵀR ≈ I：{np.allclose(r.T @ r, np.eye(3))}，det = {np.linalg.det(r):.3f}")

## 3. 代码里才看得到的

1. `fill_unified` 返回值和掩码两个数组；掩码和值必须一起传给损失，只传值就等于"空格填 0 也算分"。
2. `rot6d_to_matrix` 输入的两列不必正交，代码先归一化再减投影；输出的三列两两点积为 0。
3. RDT 表里格子是左闭右开（[0, 10) 是 0–9），代码按 Python 的 `range` 写，正好对上。

## 4. 自己改一改

- 给 `unified_slots` 加一种"7 自由度双臂 + 底盘"的机器人，数一数用了几格。
- 把 `masked_mse` 的掩码全设成 1，比较空格填 0 时损失变成多少。
- 给 `rot6d_to_matrix` 喂两列平行的向量，看会发生什么（Gram–Schmidt 什么时候失效）。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 插座各国不一样，万能插座每孔标名；
> 报税表每栏一义，没有的就空着不算分；
> 翻译员是同一个，纸条说明是哪国人。